In [45]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "./data"  
for dirname, _, filenames in os.walk(DATA_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./data/train.csv
./data/test.csv
./data/data_description.txt
./data/sample_submission.csv


In [46]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))

In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(df, test_size=0.2, random_state=42)

df.shape, X_train.shape, X_test.shape

((1460, 81), (1168, 81), (292, 81))

In [48]:
X_train.dtypes

Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
SalePrice          int64
Length: 81, dtype: object

In [49]:
X_train.isna().mean().sort_values(ascending=False).head(20)

PoolQC          0.994863
MiscFeature     0.960616
Alley           0.936644
Fence           0.800514
MasVnrType      0.584760
FireplaceQu     0.468322
LotFrontage     0.185788
GarageYrBlt     0.054795
GarageCond      0.054795
GarageType      0.054795
GarageFinish    0.054795
GarageQual      0.054795
BsmtQual        0.023973
BsmtCond        0.023973
BsmtFinType2    0.023973
BsmtFinType1    0.023973
BsmtExposure    0.023973
MasVnrArea      0.005137
Electrical      0.000856
Id              0.000000
dtype: float64

In [50]:
na_frac = X_train.isna().mean()
HIGH_NA_THRESHOLD = 0.80
cols_to_drop = na_frac[na_frac > HIGH_NA_THRESHOLD].index.tolist()
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)
print("Dropped columns:", cols_to_drop)
print("Remaining shape:", X_train.shape, X_test.shape)

Dropped columns: ['Alley', 'PoolQC', 'Fence', 'MiscFeature']
Remaining shape: (1168, 77) (292, 77)


In [51]:
X_train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
254,255,20,RL,70.0,8400,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,6,2010,WD,Normal,145000
1066,1067,60,RL,59.0,7837,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,5,2009,WD,Normal,178000
638,639,30,RL,67.0,8777,Pave,Reg,Lvl,AllPub,Inside,...,164,0,0,0,0,5,2008,WD,Normal,85000
799,800,50,RL,60.0,7200,Pave,Reg,Lvl,AllPub,Corner,...,264,0,0,0,0,6,2007,WD,Normal,175000
380,381,50,RL,50.0,5000,Pave,Reg,Lvl,AllPub,Inside,...,242,0,0,0,0,5,2010,WD,Normal,127000


In [52]:
train_ids = X_train.pop("Id")
test_ids = X_test.pop("Id")

y_train = np.log1p(X_train.pop("SalePrice"))
y_test = np.log1p(X_test.pop("SalePrice"))

In [53]:
num_cols = X_train.select_dtypes(include=[np.number]).columns
for col in num_cols:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

In [54]:
cat_cols = X_train.select_dtypes(include=["object"]).columns
for col in cat_cols:
    X_train[col] = X_train[col].fillna("None")
    X_test[col] = X_test[col].fillna("None")

In [55]:
X_train = pd.get_dummies(X_train, drop_first=False)
X_test = pd.get_dummies(X_test, drop_first=False)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [56]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((1168, 284), (1168,), (292, 284), (292,))

In [57]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X_train, y_train)

In [58]:
from sklearn.metrics import root_mean_squared_error

pred_train_log = model.predict(X_train)
rmse_train_log = root_mean_squared_error(y_train, pred_train_log)
rmse_train_log

0.09226309655224262

In [59]:
pred_test_log = model.predict(X_test)
rmse_test_log = root_mean_squared_error(y_test, pred_test_log)
rmse_test_log

0.1295601384181801

In [60]:
from sklearn.metrics import root_mean_squared_error

pred_test_price = np.expm1(pred_test_log)
actual_test_price = np.expm1(y_test)

rmse_test_dollars = root_mean_squared_error(actual_test_price, pred_test_price)
print(f"RMSE on SalePrice (hold-out, dollars): {rmse_test_dollars:,.2f}")

RMSE on SalePrice (hold-out, dollars): 22,889.60


In [61]:
pd.DataFrame({
    "actual": np.expm1(y_test),
    "predicted": np.expm1(pred_test_log),
}).head(10)

,actual,predicted
892,154500.0,153485.492964
1105,325000.0,341400.079044
413,115000.0,98649.862547
522,159000.0,164476.724578
1036,315500.0,310025.686209
614,75500.0,79460.146411
218,311500.0,247828.097422
1160,146000.0,147364.826108
649,84500.0,74762.605071
887,135500.0,141070.919057


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmse = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train), start=1):
    X_tr = X_train.iloc[tr_idx]
    X_va = X_train.iloc[va_idx]

    if isinstance(y_train, pd.Series):
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]
    else:
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

    model = LinearRegression().fit(X_tr, y_tr)
    pred_va = model.predict(X_va)
    rmse = root_mean_squared_error(y_va, pred_va)
    fold_rmse.append(rmse)
    print(f"Fold {fold} — RMSE (log): {rmse:.6f}")

print(f"Mean: {np.mean(fold_rmse):.6f}, Std: {np.std(fold_rmse):.6f}")